# Numbeo Long-Run Scraper for Google Colab

This is a separate notebook built for larger scraping runs with stronger checkpointing, resume support, and Google Drive-first storage.

Important reality check: Colab can still disconnect, recycle the runtime, or stop an idle session. This notebook is designed to save progress after every completed task so you can resume instead of losing work. It does not disable Colab service limits.


In [ ]:
# Colab setup
!pip install -q beautifulsoup4 requests


## Optional: back up the Colab runtime to Google Drive

Run this when you want to save the current `/content` workspace and installed packages to Drive before the runtime resets.


In [ ]:
from google.colab import drive
import shutil, os, subprocess

# --- Safe mount (skip if already mounted) ---
mount_point = '/content/drive'
if not os.path.isdir(os.path.join(mount_point, 'MyDrive')):
    drive.mount(mount_point)  # only mounts when needed
else:
    print("Drive already mounted, skipping mount step")

# --- Paths ---
backup_dir = '/content/drive/MyDrive/colab_backup'
zip_path = '/content/colab_runtime_backup.zip'

os.makedirs(backup_dir, exist_ok=True)

# --- Remove old zip if exists ---
if os.path.exists(zip_path):
    os.remove(zip_path)

# --- Zip /content excluding Drive to avoid recursion ---
subprocess.run([
    'zip', '-r', zip_path, '/content',
    '-x', '/content/drive/*'
], check=True)

# --- Move zip to Drive ---
final_zip = os.path.join(backup_dir, 'colab_runtime_backup.zip')
if os.path.exists(final_zip):
    os.remove(final_zip)

shutil.move(zip_path, final_zip)

# --- Save environment ---
with open(os.path.join(backup_dir, 'requirements.txt'), 'w') as f:
    subprocess.run(['pip', 'freeze'], stdout=f)

print("Backup complete:", backup_dir)


## Google Drive output and long-run notes

This notebook writes directly into Google Drive using a country-first folder structure:

```text
MyDrive/numbeo_runs/<run_name>/
|-- progress.json
|-- run_summary.json
|-- logs/run.log
|-- countries/
    |-- malaysia/
        |-- cost-of-living/
            |-- country.json
            |-- index.json
            |-- cities/
                |-- kuala-lumpur.json
        |-- crime/
            |-- country.json
```

That means each finished country/category task is saved immediately, so reruns can skip completed work and continue from the next task.


In [ ]:
from google.colab import drive
from pathlib import Path
import os

mount_point = '/content/drive'
if not os.path.isdir(os.path.join(mount_point, 'MyDrive')):
    drive.mount(mount_point)
else:
    print("Drive already mounted")

DEFAULT_DRIVE_ROOT = '/content/drive/MyDrive/numbeo_runs'
Path(DEFAULT_DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
print(f"Drive output root ready: {DEFAULT_DRIVE_ROOT}")


In [ ]:
# -*- coding: utf-8 -*-
"""Scrape Numbeo country and city tables into JSON.

Examples:
    python scrape.py --category health-care --countries Malaysia Singapore --city-limit 2
    python scrape.py --category cost-of-living --countries France Japan --output results.json
"""

from __future__ import annotations

import argparse
import json
import random
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Any
from urllib.parse import quote_plus

import requests
from bs4 import BeautifulSoup


COUNTRY_LIST_URL = "https://www.numbeo.com/cost-of-living/"
CATEGORIES = {
    "cost-of-living": "https://www.numbeo.com/cost-of-living/",
    "crime": "https://www.numbeo.com/crime/",
    "health-care": "https://www.numbeo.com/health-care/",
    "pollution": "https://www.numbeo.com/pollution/",
    "property-prices": "https://www.numbeo.com/property-investment/",
    "quality-of-life": "https://www.numbeo.com/quality-of-life/",
    "traffic": "https://www.numbeo.com/traffic/",
}
ALL_CATEGORY_CHOICE = "all"
RATING_LABELS = {"Very Low", "Low", "Moderate", "High", "Very High"}
CURRENCY_FIXES = {
    "â‚¬": "€",
    "Â¥": "¥",
    "Ą": "¥",
}


class NumbeoScrapeError(RuntimeError):
    """Raised when a page cannot be fetched after all retries."""


def clean_text(value: str) -> str:
    """Normalize whitespace from table cells."""
    cleaned = " ".join(value.split())
    for bad, good in CURRENCY_FIXES.items():
        cleaned = cleaned.replace(bad, good)
    return cleaned


def parse_number(value: str) -> float | None:
    cleaned = value.replace(",", "").strip()
    try:
        return float(cleaned)
    except ValueError:
        return None


def split_number_and_suffix(value: str) -> tuple[float | None, str | None]:
    match = re.match(r"^\s*([+-]?\d+(?:,\d{3})*(?:\.\d+)?)\s*(.*?)\s*$", value)
    if not match:
        return None, None

    number = parse_number(match.group(1))
    suffix = match.group(2).strip()

    if number is None:
        return None, suffix or None

    return number, suffix or None


def parse_range(value: str, unit: str | None = None) -> dict[str, Any]:
    parts = re.split(r"\s+-\s+", value, maxsplit=1)
    range_data: dict[str, Any] = {"display": value}

    if len(parts) != 2:
        return range_data

    low, low_unit = split_number_and_suffix(parts[0])
    high, high_unit = split_number_and_suffix(parts[1])

    if low is not None:
        range_data["min"] = low
    if high is not None:
        range_data["max"] = high

    range_unit = unit or low_unit or high_unit
    if range_unit:
        range_data["unit"] = range_unit

    return range_data


def normalize_metric(name: str, value_display: str) -> dict[str, Any]:
    metric: dict[str, Any] = {
        "name": clean_text(name).rstrip(":"),
        "value_display": clean_text(value_display),
    }
    value_text = metric["value_display"]

    if value_text.endswith("%"):
        metric["unit"] = "%"
        value_text = value_text[:-1]

    value, suffix = split_number_and_suffix(value_text)
    if value is not None:
        metric["value"] = value
    if suffix:
        metric["unit"] = metric.get("unit") or suffix

    return metric


def normalized_cell(value: str) -> Any:
    cleaned = clean_text(value)
    number = parse_number(cleaned.rstrip("%"))
    if number is None:
        return cleaned
    if cleaned.endswith("%"):
        return {"value": number, "unit": "%", "display": cleaned}
    return number


def slugify(value: str) -> str:
    slug = re.sub(r"[^a-z0-9]+", "-", value.lower()).strip("-")
    return slug or "item"


class TableExtractor:
    """Extract a Numbeo data table into JSON-friendly sections."""

    def __init__(self, page: BeautifulSoup, table: Any | None = None, title: str = "") -> None:
        self.table = table or page.find("table", {"class": "data_wide_table"})
        self.title = title

    def extract(self) -> dict[str, list[dict[str, Any]]] | None:
        if not self.table:
            return None

        data: dict[str, list[dict[str, Any]]] = {}
        section = self.title or "Summary"

        for row in self.table.find_all("tr"):
            header = row.find("th")
            if header:
                section = clean_text(header.get_text(" "))
                data.setdefault(section, [])
                continue

            cells = [clean_text(cell.get_text(" ")) for cell in row.find_all("td")]
            cells = [cell for cell in cells if cell]
            if not cells:
                continue

            data.setdefault(section, []).append(self._normalize_row(cells))

        return data or None

    @staticmethod
    def _normalize_row(cells: list[str]) -> dict[str, Any]:
        row: dict[str, Any] = {"raw": {"cells": cells}}

        if cells:
            row["item"] = cells[0]

        if len(cells) >= 2:
            row["value_display"] = cells[1]
            value, suffix = split_number_and_suffix(cells[1])

            if value is not None:
                row["value"] = value
            else:
                row["value"] = cells[1]

            if suffix and suffix not in RATING_LABELS:
                row["unit"] = suffix

            if len(cells) >= 3 and cells[2] in RATING_LABELS:
                row["rating"] = cells[2]
            elif len(cells) >= 3:
                row["range"] = parse_range(cells[2], row.get("unit"))
            else:
                row.pop("range", None)

        if len(cells) >= 3:
            row["raw"]["range_display"] = cells[2]

        if len(cells) > 3:
            row["extra"] = cells[3:]

        return row


def nearest_heading(table: Any) -> str:
    node = table.previous_sibling
    while node:
        if getattr(node, "name", None) in {"h1", "h2", "h3"}:
            return clean_text(node.get_text(" "))
        node = node.previous_sibling
    return ""


def extract_data_tables(page: BeautifulSoup) -> list[dict[str, Any]]:
    tables: list[dict[str, Any]] = []

    for index, table in enumerate(page.find_all("table", {"class": "data_wide_table"}), start=1):
        title = nearest_heading(table) or f"Data Table {index}"
        sections = TableExtractor(page, table=table, title=title).extract() or {}
        tables.append(
            {
                "title": title,
                "sections": sections,
            }
        )

    return tables


def merge_table_sections(tables: list[dict[str, Any]]) -> dict[str, list[dict[str, Any]]]:
    merged: dict[str, list[dict[str, Any]]] = {}

    for table in tables:
        for section, rows in table["sections"].items():
            merged.setdefault(section, []).extend(rows)

    return merged


def extract_indices(page: BeautifulSoup) -> list[dict[str, Any]]:
    indices: list[dict[str, Any]] = []

    for table in page.find_all("table", {"class": "table_indices"}):
        group_title = ""
        pending_name = ""

        for token in table.stripped_strings:
            text = clean_text(token)
            if not text:
                continue

            if text == "Index" or text.startswith("country data"):
                group_title = text
                continue

            if text.endswith(":"):
                pending_name = text
                continue

            if pending_name:
                metric = normalize_metric(pending_name, text)
                if group_title:
                    metric["group"] = group_title
                indices.append(metric)
                pending_name = ""

    return indices


def extract_city_rankings(page: BeautifulSoup) -> list[dict[str, Any]]:
    rankings: list[dict[str, Any]] = []

    for table in page.find_all("table", id="t2"):
        headers = [clean_text(cell.get_text(" ")) for cell in table.find_all("th")]
        rows: list[dict[str, Any]] = []

        for tr in table.find_all("tr"):
            cells = [clean_text(cell.get_text(" ")) for cell in tr.find_all("td")]
            if not cells:
                continue

            row: dict[str, Any] = {}
            for index, value in enumerate(cells):
                header = headers[index] if index < len(headers) else f"column_{index + 1}"
                row[header or f"column_{index + 1}"] = normalized_cell(value)
            rows.append(row)

        if rows:
            rankings.append(
                {
                    "title": nearest_heading(table) or "By City",
                    "columns": headers,
                    "rows": rows,
                }
            )

    return rankings


def extract_metric_tables(page: BeautifulSoup) -> list[dict[str, Any]]:
    metric_tables: list[dict[str, Any]] = []
    excluded_classes = {
        "data_wide_table",
        "languages_ref_table",
        "standard_margin",
        "stripe",
        "table_indices",
    }

    for index, table in enumerate(page.find_all("table"), start=1):
        classes = set(table.get("class") or [])
        if classes & excluded_classes or table.get("id") == "t2":
            continue

        metrics: list[dict[str, Any]] = []
        for tr in table.find_all("tr"):
            cells = [clean_text(cell.get_text(" ")) for cell in tr.find_all(["th", "td"])]
            cells = [cell for cell in cells if cell]

            if len(cells) < 2:
                continue

            metric = normalize_metric(cells[0], cells[1])
            if len(cells) >= 3 and cells[2] in RATING_LABELS:
                metric["rating"] = cells[2]
            elif len(cells) >= 3:
                metric["extra"] = cells[2:]
            metrics.append(metric)

        if metrics:
            metric_tables.append(
                {
                    "title": nearest_heading(table) or f"Metric Table {index}",
                    "metrics": metrics,
                }
            )

    return metric_tables


def merge_metrics(metric_tables: list[dict[str, Any]]) -> list[dict[str, Any]]:
    metrics: list[dict[str, Any]] = []

    for table in metric_tables:
        for metric in table["metrics"]:
            metric_with_group = dict(metric)
            metric_with_group.setdefault("group", table["title"])
            metrics.append(metric_with_group)

    return metrics


class NumbeoScraper:
    def __init__(
        self,
        category: str,
        *,
        city_limit: int = 0,
        retries: int = 4,
        delay: float = 2.0,
        timeout: int = 20,
    ) -> None:
        if category not in CATEGORIES:
            raise ValueError(f"Unsupported category: {category}")

        self.category = category
        self.base_url = CATEGORIES[category]
        self.city_limit = city_limit
        self.retries = retries
        self.delay = delay
        self.timeout = timeout
        self.session = requests.Session()
        self.session.headers.update(
            {
                "User-Agent": (
                    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                    "AppleWebKit/537.36 (KHTML, like Gecko) "
                    "Chrome/120.0.0.0 Safari/537.36"
                ),
                "Accept-Language": "en-US,en;q=0.9",
            }
        )

    def scrape_country(self, country: str) -> dict[str, Any]:
        country_url = self._country_url(country)
        page = self._get_soup(country_url)
        tables = extract_data_tables(page)
        metric_tables = extract_metric_tables(page)
        result: dict[str, Any] = {
            "source_url": country_url,
            "indices": extract_indices(page),
            "metrics": merge_metrics(metric_tables),
            "metric_tables": metric_tables,
            "data": merge_table_sections(tables),
            "tables": tables,
            "city_rankings": extract_city_rankings(page),
            "cities": {},
        }

        cities = self._extract_cities(page)
        if self.city_limit > 0:
            for city in cities[: self.city_limit]:
                print(f"  > Scraping {country} / {city}")
                result["cities"][city] = self.scrape_city(country, city)

        return result

    def scrape_city(self, country: str, city: str) -> dict[str, Any]:
        city_url = self._city_url(country, city)
        page = self._get_soup(city_url)
        tables = extract_data_tables(page)
        metric_tables = extract_metric_tables(page)
        return {
            "source_url": city_url,
            "indices": extract_indices(page),
            "metrics": merge_metrics(metric_tables),
            "metric_tables": metric_tables,
            "data": merge_table_sections(tables),
            "tables": tables,
            "city_rankings": extract_city_rankings(page),
        }

    def _get_soup(self, url: str) -> BeautifulSoup:
        response = self._get_page(url)
        return BeautifulSoup(response.content, "html.parser")

    def _get_page(self, url: str) -> requests.Response:
        backoff = max(self.delay, 1.0)
        last_error: Exception | None = None

        for attempt in range(1, self.retries + 1):
            try:
                time.sleep(backoff + random.uniform(0.25, 1.0))
                response = self.session.get(url, timeout=self.timeout)

                if response.status_code == 200:
                    return response

                if response.status_code == 429:
                    print(f"Rate limited by Numbeo; retrying attempt {attempt}/{self.retries}")
                    backoff *= 2
                    continue

                response.raise_for_status()
            except requests.RequestException as exc:
                last_error = exc
                print(f"Request failed for {url}: {exc}")
                backoff *= 2

        raise NumbeoScrapeError(f"Could not fetch {url}") from last_error

    def _extract_cities(self, page: BeautifulSoup) -> list[str]:
        form = page.find("form", {"class": "standard_margin"})
        if not form:
            return []

        cities: list[str] = []
        for option in form.find_all("option"):
            value = option.get("value")
            if value and value.strip():
                cities.append(value.strip())
        return cities

    def _country_url(self, country: str) -> str:
        return f"{self.base_url}country_result.jsp?country={quote_plus(country)}"

    def _city_url(self, country: str, city: str) -> str:
        return (
            f"{self.base_url}city_result.jsp?"
            f"country={quote_plus(country)}&city={quote_plus(city)}"
        )


def get_available_countries(timeout: int = 20) -> list[str]:
    response = requests.get(
        COUNTRY_LIST_URL,
        headers={
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 (KHTML, like Gecko) "
                "Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept-Language": "en-US,en;q=0.9",
        },
        timeout=timeout,
    )
    response.raise_for_status()

    page = BeautifulSoup(response.content, "html.parser")
    country_form = page.find("form", action=re.compile(r"country_result\.jsp"))
    if not country_form:
        raise NumbeoScrapeError("Could not find Numbeo country selector.")

    countries: list[str] = []
    for option in country_form.find_all("option"):
        value = option.get("value")
        if value and value.strip():
            countries.append(value.strip())

    return sorted(dict.fromkeys(countries))


class Extract_table(TableExtractor):
    """Backward-compatible name used by the original project."""


class API:
    """Backward-compatible wrapper around NumbeoScraper."""

    def __init__(self, base_url: str, country: str, city_limit: int = 0) -> None:
        category = self._category_from_url(base_url)
        self.country = country
        self.scraper = NumbeoScraper(category, city_limit=city_limit)
        payload = self.scraper.scrape_country(country)
        self.result = {country: self._legacy_payload(payload)}

    def get_result(self) -> dict[str, Any]:
        return self.result.get(self.country, {})

    @staticmethod
    def _legacy_payload(payload: dict[str, Any]) -> dict[str, Any]:
        data = dict(payload.get("data", {}))
        cities = payload.get("cities", {})
        if cities:
            data["child"] = {
                city: city_payload.get("data", {}) for city, city_payload in cities.items()
            }
        return data

    @staticmethod
    def _category_from_url(base_url: str) -> str:
        for category, url in CATEGORIES.items():
            if category in base_url or url.rstrip("/") in base_url.rstrip("/"):
                return category
        return "cost-of-living"


def write_json(path: str, payload: dict[str, Any]) -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as output:
        json.dump(payload, output, indent=2, ensure_ascii=False)
        output.write("\n")
    print(f"\n[Done] Data saved to {path}")


def read_json_if_exists(path: str | Path) -> dict[str, Any] | None:
    file_path = Path(path)
    if not file_path.exists() or not file_path.is_file():
        return None

    try:
        with file_path.open("r", encoding="utf-8") as input_file:
            payload = json.load(input_file)
        if isinstance(payload, dict):
            return payload
    except (OSError, json.JSONDecodeError):
        return None

    return None


def split_output_root(output: str) -> Path:
    path = Path(output)
    return path.with_suffix("") if path.suffix else path


def country_payload(country: str, category: str, payload: dict[str, Any], city_limit: int) -> dict[str, Any]:
    return {
        "metadata": {
            "schema_version": "2.0",
            "category": category,
            "country": country,
            "scraped_at": datetime.now(timezone.utc).isoformat(),
            "city_limit": city_limit,
        },
        "country": country,
        "category": category,
        **payload,
    }


def error_payload(exc: Exception) -> dict[str, Any]:
    return {
        "error": str(exc),
        "indices": [],
        "metrics": [],
        "data": {},
        "tables": [],
        "city_rankings": [],
        "cities": {},
    }


def split_manifest(args: argparse.Namespace, countries: list[str]) -> tuple[Path, dict[str, Any]]:
    root = split_output_root(args.output)
    root.mkdir(parents=True, exist_ok=True)
    (root / ".root").touch()
    (root / "countries").mkdir(parents=True, exist_ok=True)

    metadata: dict[str, Any] = {
        "schema_version": "2.0",
        "category": args.category,
        "scraped_at": datetime.now(timezone.utc).isoformat(),
        "city_limit": max(args.city_limit, 0),
        "all_countries": bool(args.all_countries),
        "country_count": len(countries),
    }
    if args.category == ALL_CATEGORY_CHOICE:
        metadata["categories"] = sorted(CATEGORIES)

    return root, {"metadata": metadata, "files": []}


def write_split_country_category(
    root: Path,
    manifest: dict[str, Any],
    country: str,
    category: str,
    payload: dict[str, Any],
    city_limit: int,
) -> None:
    target = root / "countries" / slugify(country) / f"{slugify(category)}.json"
    write_json(str(target), country_payload(country, category, payload, city_limit))
    entry = {
        "country": country,
        "category": category,
        "path": target.relative_to(root).as_posix(),
    }
    manifest["files"] = [
        file_entry
        for file_entry in manifest["files"]
        if not (file_entry.get("country") == country and file_entry.get("category") == category)
    ]
    manifest["files"].append(entry)
    write_json(str(root / "manifest.json"), manifest)


def write_split_outputs(output: str, results: dict[str, Any]) -> None:
    root = split_output_root(output)
    root.mkdir(parents=True, exist_ok=True)
    manifest: dict[str, Any] = {
        "metadata": dict(results.get("metadata", {})),
        "files": [],
    }
    city_limit = int(results.get("metadata", {}).get("city_limit", 0))
    root_marker = root / ".root"
    root_marker.touch()

    countries_root = root / "countries"
    countries_root.mkdir(parents=True, exist_ok=True)

    for country, country_data in results.get("countries", {}).items():
        country_dir = countries_root / slugify(country)
        country_dir.mkdir(parents=True, exist_ok=True)

        if "categories" in country_data:
            for category, payload in country_data["categories"].items():
                target = country_dir / f"{slugify(category)}.json"
                write_json(str(target), country_payload(country, category, payload, city_limit))
                manifest["files"].append(
                    {
                        "country": country,
                        "category": category,
                        "path": target.relative_to(root).as_posix(),
                    }
                )
        else:
            category = results.get("metadata", {}).get("category", "unknown")
            target = country_dir / f"{slugify(category)}.json"
            write_json(str(target), country_payload(country, category, country_data, city_limit))
            manifest["files"].append(
                {
                    "country": country,
                    "category": category,
                    "path": target.relative_to(root).as_posix(),
                }
            )

    write_json(str(root / "manifest.json"), manifest)


def has_payload_data(payload: dict[str, Any] | None) -> bool:
    if not payload:
        return False

    return any(
        payload.get(key)
        for key in ("indices", "metrics", "data", "tables", "city_rankings", "cities")
    )


def split_output_target(output: str, country: str, category: str) -> Path:
    root = split_output_root(output)
    return root / "countries" / slugify(country) / f"{slugify(category)}.json"


def existing_payload_from_output(
    args: argparse.Namespace, country: str, category: str
) -> dict[str, Any] | None:
    if args.split_output:
        for candidate in (
            split_output_target(args.output, country, category),
            split_output_root(args.output) / slugify(country) / f"{slugify(category)}.json",
        ):
            split_payload = read_json_if_exists(candidate)
            if split_payload and has_payload_data(split_payload):
                return split_payload
        return None

    combined_payload = read_json_if_exists(args.output)
    if not combined_payload:
        return None

    countries = combined_payload.get("countries", {})
    country_payload = countries.get(country)
    if not isinstance(country_payload, dict):
        return None

    if combined_payload.get("metadata", {}).get("category") == ALL_CATEGORY_CHOICE:
        category_payload = country_payload.get("categories", {}).get(category)
        if isinstance(category_payload, dict) and has_payload_data(category_payload):
            return category_payload
        return None

    if combined_payload.get("metadata", {}).get("category") == category and has_payload_data(country_payload):
        return country_payload

    return None


def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(description="Scrape Numbeo tables into JSON.")
    parser.add_argument(
        "--category",
        choices=[ALL_CATEGORY_CHOICE, *sorted(CATEGORIES)],
        default="cost-of-living",
        help="Numbeo section to scrape.",
    )
    parser.add_argument(
        "--countries",
        nargs="+",
        help="Country names to scrape, for example: Malaysia Singapore Australia",
    )
    parser.add_argument(
        "--all-countries",
        action="store_true",
        help="Scrape every country currently listed by Numbeo.",
    )
    parser.add_argument(
        "--country-limit",
        type=int,
        default=0,
        help="Limit number of countries after discovery. Useful for testing --all-countries.",
    )
    parser.add_argument(
        "--city-limit",
        type=int,
        default=0,
        help="Maximum cities to scrape per country. Use 0 for country-level data only.",
    )
    parser.add_argument(
        "--output",
        default="results.json",
        help="JSON file to write, or output folder base when --split-output is used.",
    )
    parser.add_argument(
        "--split-output",
        action="store_true",
        help="Write one JSON file per country/category plus a manifest instead of one combined file.",
    )
    parser.add_argument(
        "--delay",
        type=float,
        default=2.0,
        help="Base delay between requests in seconds.",
    )
    return parser.parse_args()


def scrape_single_category(args: argparse.Namespace, countries: list[str]) -> dict[str, Any]:
    scraper = NumbeoScraper(
        args.category,
        city_limit=max(args.city_limit, 0),
        delay=max(args.delay, 0),
    )

    results: dict[str, Any] = {
        "metadata": {
            "schema_version": "2.0",
            "category": args.category,
            "scraped_at": datetime.now(timezone.utc).isoformat(),
            "city_limit": max(args.city_limit, 0),
            "all_countries": bool(args.all_countries),
            "country_count": len(countries),
        },
        "countries": {},
    }

    for country in countries:
        existing_payload = existing_payload_from_output(args, country, args.category)
        if existing_payload:
            print(f"Skipping {country}; found existing {args.category} data.")
            results["countries"][country] = existing_payload
            continue

        print(f"Processing {country}...")
        try:
            results["countries"][country] = scraper.scrape_country(country)
        except NumbeoScrapeError as exc:
            results["countries"][country] = error_payload(exc)

    return results


def scrape_single_category_split(args: argparse.Namespace, countries: list[str]) -> dict[str, Any]:
    root, manifest = split_manifest(args, countries)
    scraper = NumbeoScraper(
        args.category,
        city_limit=max(args.city_limit, 0),
        delay=max(args.delay, 0),
    )

    for country in countries:
        existing_payload = existing_payload_from_output(args, country, args.category)
        if existing_payload:
            print(f"Skipping {country}; found existing {args.category} data.")
            write_split_country_category(
                root,
                manifest,
                country,
                args.category,
                existing_payload,
                max(args.city_limit, 0),
            )
            continue

        print(f"Processing {country}...")
        try:
            payload = scraper.scrape_country(country)
        except NumbeoScrapeError as exc:
            payload = error_payload(exc)

        write_split_country_category(
            root,
            manifest,
            country,
            args.category,
            payload,
            max(args.city_limit, 0),
        )

    return manifest


def scrape_all_categories(args: argparse.Namespace, countries: list[str]) -> dict[str, Any]:
    category_names = sorted(CATEGORIES)
    results: dict[str, Any] = {
        "metadata": {
            "schema_version": "2.0",
            "category": ALL_CATEGORY_CHOICE,
            "categories": category_names,
            "scraped_at": datetime.now(timezone.utc).isoformat(),
            "city_limit": max(args.city_limit, 0),
            "all_countries": bool(args.all_countries),
            "country_count": len(countries),
        },
        "countries": {},
    }

    for country in countries:
        print(f"Processing {country}...")
        results["countries"][country] = {"categories": {}}

        for category in category_names:
            existing_payload = existing_payload_from_output(args, country, category)
            if existing_payload:
                print(f"  > {category} (skipped; already exists)")
                results["countries"][country]["categories"][category] = existing_payload
                continue

            print(f"  > {category}")
            scraper = NumbeoScraper(
                category,
                city_limit=max(args.city_limit, 0),
                delay=max(args.delay, 0),
            )
            try:
                results["countries"][country]["categories"][category] = scraper.scrape_country(country)
            except NumbeoScrapeError as exc:
                results["countries"][country]["categories"][category] = error_payload(exc)

    return results


def scrape_all_categories_split(args: argparse.Namespace, countries: list[str]) -> dict[str, Any]:
    root, manifest = split_manifest(args, countries)
    category_names = sorted(CATEGORIES)

    for country in countries:
        print(f"Processing {country}...")

        for category in category_names:
            existing_payload = existing_payload_from_output(args, country, category)
            if existing_payload:
                print(f"  > {category} (skipped; already exists)")
                write_split_country_category(
                    root,
                    manifest,
                    country,
                    category,
                    existing_payload,
                    max(args.city_limit, 0),
                )
                continue

            print(f"  > {category}")
            scraper = NumbeoScraper(
                category,
                city_limit=max(args.city_limit, 0),
                delay=max(args.delay, 0),
            )
            try:
                payload = scraper.scrape_country(country)
            except NumbeoScrapeError as exc:
                payload = error_payload(exc)

            write_split_country_category(
                root,
                manifest,
                country,
                category,
                payload,
                max(args.city_limit, 0),
            )

    return manifest


def main() -> None:
    args = parse_args()
    countries = args.countries

    if args.all_countries:
        print("Discovering countries from Numbeo...")
        countries = get_available_countries()
        if args.country_limit > 0:
            countries = countries[: args.country_limit]
        print(f"Discovered {len(countries)} countries.")
    elif not countries:
        raw_countries = input("Enter countries separated by commas: ")
        countries = [country.strip() for country in raw_countries.split(",") if country.strip()]

    if args.split_output and args.category == ALL_CATEGORY_CHOICE:
        scrape_all_categories_split(args, countries)
    elif args.split_output:
        scrape_single_category_split(args, countries)
    elif args.category == ALL_CATEGORY_CHOICE:
        results = scrape_all_categories(args, countries)
        write_json(args.output, results)
    else:
        results = scrape_single_category(args, countries)
        write_json(args.output, results)




In [ ]:
def write_json(path: str, payload: dict[str, Any]) -> None:
    output_path = Path(path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with output_path.open("w", encoding="utf-8") as output:
        json.dump(payload, output, indent=2, ensure_ascii=False)
        output.write("\n")

import json
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

RUN_SCHEMA_VERSION = "3.1"
PROGRESS_LOCK = threading.Lock()

SCRAPER_RUNTIME_SETTINGS = {
    "proxy_url": "",
    "request_timeout_seconds": 30,
    "min_retries": 6,
}

_ORIGINAL_RUNTIME_SCRAPER_INIT = NumbeoScraper.__init__


def configure_runtime_scraper(proxy_url: str = "", request_timeout_seconds: int = 30, min_retries: int = 6) -> None:
    SCRAPER_RUNTIME_SETTINGS["proxy_url"] = (proxy_url or "").strip()
    SCRAPER_RUNTIME_SETTINGS["request_timeout_seconds"] = max(int(request_timeout_seconds), 1)
    SCRAPER_RUNTIME_SETTINGS["min_retries"] = max(int(min_retries), 1)


def _patched_runtime_scraper_init(self, category: str, *, city_limit: int = 0, retries: int = 4, delay: float = 2.0, timeout: int = 20) -> None:
    _ORIGINAL_RUNTIME_SCRAPER_INIT(self, category, city_limit=city_limit, retries=retries, delay=delay, timeout=timeout)
    proxy_url = SCRAPER_RUNTIME_SETTINGS["proxy_url"]
    if proxy_url:
        self.session.proxies.update({"http": proxy_url, "https": proxy_url})
    self.timeout = max(self.timeout, SCRAPER_RUNTIME_SETTINGS["request_timeout_seconds"])
    self.retries = max(self.retries, SCRAPER_RUNTIME_SETTINGS["min_retries"])


NumbeoScraper.__init__ = _patched_runtime_scraper_init
configure_runtime_scraper()


def pause_after_country_batch(processed_countries: int, pause_every_n_countries: int, pause_seconds: float) -> None:
    if pause_every_n_countries <= 0 or pause_seconds <= 0:
        return
    if processed_countries % pause_every_n_countries != 0:
        return
    print(f"Pausing for {pause_seconds} seconds after {processed_countries} countries.")
    time.sleep(pause_seconds)


def task_with_retry(task_callable, label: str, failed_retry_rounds: int, delay_seconds: float):
    max_attempts = max(1, int(failed_retry_rounds) + 1)
    last_error = None
    for attempt in range(1, max_attempts + 1):
        try:
            return task_callable(), attempt
        except Exception as exc:
            last_error = exc
            if attempt < max_attempts:
                cooldown = max(2.0, float(delay_seconds)) * attempt
                print(f"Retrying {label}; attempt {attempt + 1}/{max_attempts} after error: {exc}")
                time.sleep(cooldown)
    raise last_error


def drive_run_root(drive_root: str, run_name: str) -> Path:
    return Path(drive_root) / slugify(run_name)

def ensure_drive_run_root(drive_root: str, run_name: str) -> Path:
    root = drive_run_root(drive_root, run_name)
    (root / "countries").mkdir(parents=True, exist_ok=True)
    (root / "logs").mkdir(parents=True, exist_ok=True)
    (root / ".root").touch()
    return root

def category_root(root: Path, country: str, category: str) -> Path:
    return root / "countries" / slugify(country) / slugify(category)

def country_json_path(root: Path, country: str, category: str) -> Path:
    return category_root(root, country, category) / "country.json"

def category_index_path(root: Path, country: str, category: str) -> Path:
    return category_root(root, country, category) / "index.json"

def city_json_path(root: Path, country: str, category: str, city: str) -> Path:
    return category_root(root, country, category) / "cities" / f"{slugify(city)}.json"

def progress_path(root: Path) -> Path:
    return root / "progress.json"

def run_summary_path(root: Path) -> Path:
    return root / "run_summary.json"

def run_log_path(root: Path) -> Path:
    return root / "logs" / "run.log"

def read_json_path(path: Path) -> dict[str, Any] | None:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except json.JSONDecodeError:
        return None

def append_log(root: Path, message: str) -> None:
    timestamp = datetime.now(timezone.utc).isoformat()
    with run_log_path(root).open("a", encoding="utf-8") as handle:
        handle.write(f"[{timestamp}] {message}\n")

def payload_city_names(payload: dict[str, Any]) -> list[str]:
    return sorted((payload.get("cities") or {}).keys())

def payload_row_count(payload: dict[str, Any]) -> int:
    return sum(len(rows) for rows in (payload.get("data") or {}).values())

def payload_table_count(payload: dict[str, Any]) -> int:
    return len(payload.get("tables") or [])

def payload_metric_count(payload: dict[str, Any]) -> int:
    return len(payload.get("metrics") or [])

def payload_index_count(payload: dict[str, Any]) -> int:
    return len(payload.get("indices") or [])

def payload_ranking_count(payload: dict[str, Any]) -> int:
    return len(payload.get("city_rankings") or [])

def strip_cities(payload: dict[str, Any]) -> dict[str, Any]:
    cleaned = dict(payload)
    cleaned["cities"] = {}
    return cleaned

def build_category_index(country: str, category: str, payload: dict[str, Any], root: Path) -> dict[str, Any]:
    city_names = payload_city_names(payload)
    return {
        "schema_version": RUN_SCHEMA_VERSION,
        "country": country,
        "category": category,
        "source_url": payload.get("source_url"),
        "error": payload.get("error"),
        "counts": {
            "indices": payload_index_count(payload),
            "metrics": payload_metric_count(payload),
            "tables": payload_table_count(payload),
            "data_rows": payload_row_count(payload),
            "city_rankings": payload_ranking_count(payload),
            "cities": len(city_names),
        },
        "files": {
            "country": country_json_path(root, country, category).relative_to(root).as_posix(),
            "cities": [city_json_path(root, country, category, city).relative_to(root).as_posix() for city in city_names],
        },
    }

def write_structured_payload(root: Path, country: str, category: str, payload: dict[str, Any], city_limit: int) -> None:
    cat_root = category_root(root, country, category)

    base_payload = country_payload(country, category, strip_cities(payload), city_limit)
    write_json(str(country_json_path(root, country, category)), base_payload)

    city_items = sorted((payload.get("cities") or {}).items())
    if city_items:
        (cat_root / "cities").mkdir(parents=True, exist_ok=True)

    for city, city_payload in city_items:
        city_doc = {
            "metadata": {
                "schema_version": RUN_SCHEMA_VERSION,
                "country": country,
                "category": category,
                "city": city,
                "scraped_at": datetime.now(timezone.utc).isoformat(),
            },
            "payload": city_payload,
        }
        write_json(str(city_json_path(root, country, category, city)), city_doc)

    write_json(str(category_index_path(root, country, category)), build_category_index(country, category, payload, root))

def default_progress(run_name: str, drive_root: str, categories: list[str], countries: list[str], city_limit: int) -> dict[str, Any]:
    return {
        "schema_version": RUN_SCHEMA_VERSION,
        "run": {
            "name": run_name,
            "drive_root": drive_root,
            "categories": categories,
            "country_count": len(countries),
            "city_limit": city_limit,
            "started_at": datetime.now(timezone.utc).isoformat(),
            "updated_at": datetime.now(timezone.utc).isoformat(),
        },
        "tasks": [],
    }

def upsert_task(progress: dict[str, Any], entry: dict[str, Any]) -> dict[str, Any]:
    progress.setdefault("tasks", [])
    progress["tasks"] = [
        item for item in progress["tasks"]
        if not (item.get("country") == entry.get("country") and item.get("category") == entry.get("category"))
    ]
    progress["tasks"].append(entry)
    progress["tasks"].sort(key=lambda item: (item.get("country", ""), item.get("category", "")))
    progress["run"]["updated_at"] = datetime.now(timezone.utc).isoformat()
    return progress

def save_progress(root: Path, progress: dict[str, Any]) -> None:
    write_json(str(progress_path(root)), progress)

def load_or_create_progress(root: Path, run_name: str, drive_root: str, categories: list[str], countries: list[str], city_limit: int) -> dict[str, Any]:
    existing = read_json_path(progress_path(root))
    if existing:
        return existing
    created = default_progress(run_name, drive_root, categories, countries, city_limit)
    save_progress(root, created)
    return created

def task_status(progress: dict[str, Any], country: str, category: str) -> str | None:
    for item in progress.get("tasks", []):
        if item.get("country") == country and item.get("category") == category:
            return item.get("status")
    return None

def task_is_complete(root: Path, progress: dict[str, Any], country: str, category: str) -> bool:
    return country_json_path(root, country, category).exists() and task_status(progress, country, category) == "completed"

def task_entry(root: Path, country: str, category: str, status: str, payload: dict[str, Any] | None = None, error_message: str | None = None) -> dict[str, Any]:
    entry = {
        "country": country,
        "category": category,
        "status": status,
        "updated_at": datetime.now(timezone.utc).isoformat(),
        "path": country_json_path(root, country, category).relative_to(root).as_posix(),
    }
    if payload is not None:
        entry["counts"] = {
            "indices": payload_index_count(payload),
            "metrics": payload_metric_count(payload),
            "tables": payload_table_count(payload),
            "data_rows": payload_row_count(payload),
            "city_rankings": payload_ranking_count(payload),
            "cities": len(payload_city_names(payload)),
        }
    if error_message:
        entry["error"] = error_message
    return entry

def build_run_summary(root: Path, progress: dict[str, Any]) -> dict[str, Any]:
    countries: dict[str, dict[str, Any]] = {}
    for task in progress.get("tasks", []):
        countries.setdefault(task["country"], {"categories": {}})
        countries[task["country"]]["categories"][task["category"]] = {
            "status": task.get("status"),
            "path": task.get("path"),
            "counts": task.get("counts", {}),
            "error": task.get("error"),
        }
    summary = {
        "schema_version": RUN_SCHEMA_VERSION,
        "generated_at": datetime.now(timezone.utc).isoformat(),
        "run": progress.get("run", {}),
        "countries": countries,
    }
    write_json(str(run_summary_path(root)), summary)
    return summary

def discover_notebook_countries(all_countries: bool, countries: list[str], country_limit: int) -> list[str]:
    if all_countries:
        discovered = get_available_countries()
        if country_limit > 0:
            discovered = discovered[:country_limit]
        return discovered
    return [country for country in countries if country]

def discover_notebook_categories(selected_categories: list[str]) -> list[str]:
    if not selected_categories:
        return sorted(CATEGORIES)
    if len(selected_categories) == 1 and selected_categories[0] == ALL_CATEGORY_CHOICE:
        return sorted(CATEGORIES)
    return [category for category in selected_categories if category in CATEGORIES]

def pending_tasks(root: Path, progress: dict[str, Any], countries: list[str], categories: list[str], overwrite_existing: bool) -> tuple[list[tuple[str, str]], list[tuple[str, str]]]:
    todo: list[tuple[str, str]] = []
    skipped: list[tuple[str, str]] = []
    for country in countries:
        for category in categories:
            if not overwrite_existing and task_is_complete(root, progress, country, category):
                skipped.append((country, category))
                continue
            todo.append((country, category))
    return todo, skipped

def scrape_task(country: str, category: str, city_limit: int, delay_seconds: float) -> dict[str, Any]:
    scraper = NumbeoScraper(category, city_limit=max(city_limit, 0), delay=max(delay_seconds, 0))
    payload = scraper.scrape_country(country)
    return payload

def run_long_scrape(
    drive_root: str,
    run_name: str,
    selected_categories: list[str],
    countries: list[str],
    all_countries: bool = False,
    country_limit: int = 0,
    city_limit: int = 0,
    delay_seconds: float = 1.0,
    max_workers: int = 4,
    overwrite_existing: bool = False,
    proxy_url: str = "",
    request_timeout_seconds: int = 30,
    min_retries: int = 6,
    failed_retry_rounds: int = 1,
    pause_every_n_countries: int = 10,
    pause_seconds: float = 60.0,
) -> dict[str, Any]:
    categories = discover_notebook_categories(selected_categories)
    resolved_countries = discover_notebook_countries(all_countries, countries, country_limit)
    if not resolved_countries:
        raise ValueError("No countries selected for scraping.")
    if not categories:
        raise ValueError("No valid categories selected for scraping.")

    configure_runtime_scraper(proxy_url=proxy_url, request_timeout_seconds=request_timeout_seconds, min_retries=min_retries)
    root = ensure_drive_run_root(drive_root, run_name)
    progress = load_or_create_progress(root, run_name, drive_root, categories, resolved_countries, city_limit)
    todo, skipped = pending_tasks(root, progress, resolved_countries, categories, overwrite_existing)

    print(f"Run root: {root}")
    print(f"Countries: {len(resolved_countries)}")
    print(f"Categories: {categories}")
    print(f"Tasks to run: {len(todo)}")
    print(f"Tasks skipped as already complete: {len(skipped)}")

    append_log(root, f"Run started with {len(todo)} pending tasks and {len(skipped)} skipped tasks.")
    completed = 0
    failed = 0
    total = len(todo)

    for country, category in skipped:
        print(f"Skipping {country} / {category}; already complete.")

    if not todo:
        summary = build_run_summary(root, progress)
        return {"root": str(root), "progress": progress, "summary": summary, "pending": 0, "completed_now": 0, "failed_now": 0}

    for country_index, country in enumerate(resolved_countries, start=1):
        pending_categories = [category for category in categories if overwrite_existing or not task_is_complete(root, progress, country, category)]
        if not pending_categories:
            pause_after_country_batch(country_index, pause_every_n_countries, pause_seconds)
            continue
        print(f"Processing {country}...")
        with ThreadPoolExecutor(max_workers=max(1, int(max_workers))) as executor:
            future_map = {executor.submit(task_with_retry, lambda country=country, category=category: scrape_task(country, category, city_limit, delay_seconds), f"{country} / {category}", failed_retry_rounds, delay_seconds): category for category in pending_categories}
            for future in as_completed(future_map):
                category = future_map[future]
                try:
                    payload, attempts_used = future.result()
                    write_structured_payload(root, country, category, payload, city_limit)
                    with PROGRESS_LOCK:
                        progress = read_json_path(progress_path(root)) or progress
                        progress = upsert_task(progress, task_entry(root, country, category, "completed", payload=payload))
                        save_progress(root, progress)
                    completed += 1
                    print(f"Completed {country} / {category} in {attempts_used} attempt(s)")
                    append_log(root, f"Completed {country} / {category} in {attempts_used} attempt(s)")
                except Exception as exc:
                    payload = error_payload(exc)
                    write_structured_payload(root, country, category, payload, city_limit)
                    with PROGRESS_LOCK:
                        progress = read_json_path(progress_path(root)) or progress
                        progress = upsert_task(progress, task_entry(root, country, category, "failed", payload=payload, error_message=str(exc)))
                        save_progress(root, progress)
                    failed += 1
                    print(f"Failed {country} / {category}: {exc}")
                    append_log(root, f"Failed {country} / {category}: {exc}")
        pause_after_country_batch(country_index, pause_every_n_countries, pause_seconds)

    progress = read_json_path(progress_path(root)) or progress
    summary = build_run_summary(root, progress)
    append_log(root, f"Run finished. completed_now={completed}, failed_now={failed}")
    return {"root": str(root), "progress": progress, "summary": summary, "pending": len(todo), "completed_now": completed, "failed_now": failed}


## Choose what to scrape

Set `all_countries = True` to discover every country currently listed by Numbeo.
Set `selected_categories = ["all"]` to scrape every supported Numbeo section.

Drive behavior:
- progress and output are written into Google Drive
- completed country/category files already in Drive are skipped automatically

Runtime controls:
- `proxy_url`: optional single proxy that you provide
- `pause_every_n_countries`: cooldown after completed country batches
- `failed_retry_rounds`: extra task-level retries on top of built-in request retries


In [ ]:
drive_root = DEFAULT_DRIVE_ROOT
run_name = "global-numbeo-long-run"

selected_categories = ["all"]  # or e.g. ["cost-of-living", "crime", "health-care"]
all_countries = False  # set True only if you want every available country
countries = []  # add your own country names when all_countries is False
country_limit = 0  # optional cap when all_countries is True
city_limit = 0  # set > 0 to also save per-city files

delay_seconds = 1.25
max_workers = 4
overwrite_existing = False
proxy_url = ""  # optional: one proxy URL that you control
request_timeout_seconds = 30
min_retries = 6
failed_retry_rounds = 1
pause_every_n_countries = 10
pause_seconds = 60.0


In [ ]:
run_result = run_long_scrape(
    drive_root=drive_root,
    run_name=run_name,
    selected_categories=selected_categories,
    countries=countries,
    all_countries=all_countries,
    country_limit=country_limit,
    city_limit=city_limit,
    delay_seconds=delay_seconds,
    max_workers=max_workers,
    overwrite_existing=overwrite_existing,
    proxy_url=proxy_url,
    request_timeout_seconds=request_timeout_seconds,
    min_retries=min_retries,
    failed_retry_rounds=failed_retry_rounds,
    pause_every_n_countries=pause_every_n_countries,
    pause_seconds=pause_seconds,
)

print(json.dumps({
    "root": run_result["root"],
    "pending": run_result["pending"],
    "completed_now": run_result["completed_now"],
    "failed_now": run_result["failed_now"],
}, indent=2))


In [ ]:
# Preview current progress and a small part of the summary.
root = Path(run_result["root"])
progress_doc = read_json_path(progress_path(root)) or {}
summary_doc = read_json_path(run_summary_path(root)) or {}

print(json.dumps(progress_doc.get("run", {}), indent=2))
print()
print(json.dumps(dict(list(summary_doc.get("countries", {}).items())[:2]), indent=2))


In [ ]:
# Optional: compact progress summary for the current Drive run.
root = Path(drive_run_root(drive_root, run_name))
progress_doc = read_json_path(progress_path(root)) or {}
tasks = progress_doc.get('tasks', [])
completed = sum(1 for task in tasks if task.get('status') == 'completed')
failed = sum(1 for task in tasks if task.get('status') == 'failed')
country_total = progress_doc.get('run', {}).get('country_count', 0)
category_total = len(progress_doc.get('run', {}).get('categories', []))
target_tasks = country_total * category_total if country_total and category_total else len(tasks)
remaining = max(target_tasks - completed - failed, 0)
print(json.dumps({
    'completed': completed,
    'failed': failed,
    'remaining': remaining,
    'target_tasks': target_tasks,
}, indent=2))


In [ ]:
# Optional: show the current Drive run folder contents.
root = Path(run_result["root"])
print(f"Listing: {root}")
if root.exists():
    for path in sorted(root.rglob('*')):
        rel = path.relative_to(root)
        marker = '/' if path.is_dir() else ''
        print(f"{rel}{marker}")
else:
    print("Run folder does not exist yet.")


In [ ]:
# Optional: rerun only failed tasks from the current Drive run.
root = Path(drive_run_root(drive_root, run_name))
progress_doc = read_json_path(progress_path(root)) or {}
failed_tasks = [task for task in progress_doc.get("tasks", []) if task.get("status") == "failed"]
failed_countries = sorted({task["country"] for task in failed_tasks})

if not failed_tasks:
    print("No failed tasks found in progress.json")
else:
    categories_to_retry = sorted({task["category"] for task in failed_tasks})
    print(f"Retrying {len(failed_tasks)} failed task(s) across {len(failed_countries)} countrie(s).")
    rerun_result = run_long_scrape(
        drive_root=drive_root,
        run_name=run_name,
        selected_categories=categories_to_retry,
        countries=failed_countries,
        all_countries=False,
        country_limit=0,
        city_limit=city_limit,
        delay_seconds=delay_seconds,
        max_workers=max_workers,
        overwrite_existing=True,
        proxy_url=proxy_url,
        request_timeout_seconds=request_timeout_seconds,
        min_retries=min_retries,
        failed_retry_rounds=failed_retry_rounds,
        pause_every_n_countries=pause_every_n_countries,
        pause_seconds=pause_seconds,
    )
    print(json.dumps({
        "root": rerun_result["root"],
        "completed_now": rerun_result["completed_now"],
        "failed_now": rerun_result["failed_now"],
    }, indent=2))


In [ ]:
# Optional: zip one completed run folder in Drive for download or archiving.
import shutil

root = Path(run_result["root"])
archive_base = str(root)
archive_path = shutil.make_archive(archive_base, "zip", root_dir=root)
print(f"Created archive: {archive_path}")


In [ ]:
# Optional: test the configured proxy before a long run.
import requests

if proxy_url.strip():
    try:
        response = requests.get(
            'https://api.ipify.org',
            proxies={'http': proxy_url, 'https': proxy_url},
            timeout=request_timeout_seconds,
        )
        response.raise_for_status()
        print(f"Proxy check succeeded. Exit IP: {response.text.strip()}")
    except Exception as exc:
        print(f"Proxy check failed: {exc}")
else:
    print("No proxy_url configured; skipping proxy test.")


## Optional: check the current Colab public IP

Run this if you want to see the current public IP address of the active runtime.


In [ ]:
!curl -s https://api.ipify.org && echo
